# Ensemble methods. Exercises


In this section we have only two exercise:

1. Find the best three classifier in the stacking method using the classifiers from scikit-learn package.

2. Build arcing arc-x4 method. 

In [7]:
%store -r data_set
%store -r labels
%store -r test_data_set
%store -r test_labels
%store -r unique_labels

## Exercise 1: Find the best three classifier in the stacking method

Please use the following classifiers:

* Linear regression,
* Nearest Neighbors,
* Linear SVM,
* Decision Tree,
* Naive Bayes,
* QDA.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

In [ ]:
def build_classifiers():
    
    # Instantiate the classifiers
    # We choose 3 as the build_stacked_classifier function expects this.
    # You might want to experiment with different combinations or all 6
    # and use a selection process if the goal was strictly "best three".
    clf1 = KNeighborsClassifier(n_neighbors=3) 
    clf2 = DecisionTreeClassifier(random_state=1)
    clf3 = GaussianNB()
    
    classifiers = [clf1, clf2, clf3]
    
    # Train the classifiers
    for clf in classifiers:
        # Ensure labels is a 1D array using ravel()
        clf.fit(data_set, labels.ravel()) 

    return classifiers

In [ ]:
# Add this import to Cell 4
from sklearn.linear_model import LogisticRegression 

# Replace the contents of Cell 6 with this:
def build_stacked_classifier(classifiers):
    output = []
    for classifier in classifiers:
        # Ensure labels is a 1D array using ravel() for consistency if needed,
        # though predict typically doesn't need it.
        output.append(classifier.predict(data_set)) 
    
    # Stack the predictions horizontally (samples, features from classifiers)
    output = np.array(output).T # Transpose to get shape (130, 3)
    
    # stacked classifier part:
    # Use Logistic Regression as the meta-classifier
    stacked_classifier = LogisticRegression(random_state=1) 
    stacked_classifier.fit(output, labels.ravel()) # Use ravel() for 1D labels
    
    # Predict on the test set using base classifiers
    test_set_output = []
    for classifier in classifiers:
        test_set_output.append(classifier.predict(test_data_set))
        
    # Stack the test predictions horizontally
    test_set_output = np.array(test_set_output).T # Transpose to get shape (n_test_samples, 3)
    
    # Make final predictions using the meta-classifier
    predicted = stacked_classifier.predict(test_set_output)
    return predicted

In [ ]:
classifiers = build_classifiers()
predicted = build_stacked_classifier(classifiers)
accuracy = accuracy_score(test_labels, predicted)
print(accuracy)

## Exercise 2: 

Use the boosting method and change the code to fullfilt the following requirements:

* the weights should be calculated as:
$w_{n}^{(t+1)}=\frac{1+ I(y_{n}\neq h_{t}(x_{n})}{\sum_{i=1}^{N}1+I(y_{n}\neq h_{t}(x_{n})}$,
* the prediction is done with a voting method.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# prepare data set

def generate_data(sample_number, feature_number, label_number):
    data_set = np.random.random_sample((sample_number, feature_number))
    labels = np.random.choice(label_number, sample_number)
    return data_set, labels

labels = 2
dimension = 2
test_set_size = 1000
train_set_size = 5000
train_set, train_labels = generate_data(train_set_size, dimension, labels)
test_set, test_labels = generate_data(test_set_size, dimension, labels)

# init weights
number_of_iterations = 10
weights = np.ones((test_set_size,)) / test_set_size


def train_model(classifier, weights):
    return classifier.fit(X=test_set, y=test_labels, sample_weight=weights)

def calculate_error(model):
    predicted = model.predict(test_set)
    I=calculate_accuracy_vector(predicted, test_labels)
    Z=np.sum(I)
    return (1+Z)/1.0

Fill the two functions below:

In [ ]:
def set_new_weights(model):
    # 1) get this stump’s predictions on the weighted data
    preds = model.predict(test_set)
    # 2) indicator 1 if misclassified, 0 otherwise
    mis = (preds != test_labels).astype(int)
    # 3) new weights ∝ (1 + mis), normalized to sum to 1
    return (1 + mis) / np.sum(1 + mis)


Train the classifier with the code below:

In [ ]:
classifier = DecisionTreeClassifier(max_depth=1, random_state=1)
classifier.fit(X=train_set, y=train_labels)
alphas = []
classifiers = []
for iteration in range(number_of_iterations):
    model = train_model(classifier, weights)
    weights = set_new_weights(model)
    classifiers.append(model)

print(weights)


validate_x, validate_label = generate_data(1, dimension, labels)

Set the validation data set:

In [ ]:
validate_x, validate_label = generate_data(1, dimension, labels)

Fill the prediction code:

In [ ]:
def get_prediction(x):
    # x: array of shape (n_samples, n_features)
    # collect each stump’s votes
    votes = np.array([clf.predict(x) for clf in classifiers])  
    # now votes.shape == (n_classifiers, n_samples)
    votes = votes.T  # -> (n_samples, n_classifiers)

    final = []
    for row in votes:
        # row is a length-n_classifiers array of labels
        vals, cnts = np.unique(row, return_counts=True)
        final.append(vals[np.argmax(cnts)])
    return np.array(final)


Test it:

In [ ]:
prediction = get_prediction(validate_x)[0]

print(prediction)